<a href="https://colab.research.google.com/github/CatherineMatangu/MyRepo_2026_Analytics2/blob/main/CreditCardFraudDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# 1. Generate Synthetic Dataset matching the 2023 Kaggle structure
# Over 550,000 records, 28 anonymized features (V1-V28), Amount, and Class
print("Generating synthetic dataset matching Kaggle 2023 structure...")
n_samples = 100000  # Using 100k for faster processing in this environment
n_features = 28
X, y = make_classification(n_samples=n_samples, n_features=n_features, n_informative=20,
                           n_redundant=5, n_clusters_per_class=2, weights=[0.99, 0.01],
                           flip_y=0.01, random_state=42)

columns = [f'V{i}' for i in range(1, n_features + 1)]
df = pd.DataFrame(X, columns=columns)
df['Amount'] = np.random.exponential(scale=100, size=n_samples)
df['Class'] = y

# 2. Data Exploration
print("\n--- Step 1: Data Exploration ---")
print(f"Dataset Shape: {df.shape}")
print("\nBasic Information:")
print(df.info())
print("\nMissing Values:")
print(df.isnull().sum().sum())

print("\nTarget Variable Distribution:")
print(df['Class'].value_counts(normalize=True))

# Visualization: Target Distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='Class', data=df)
plt.title('Distribution of Fraudulent vs Non-Fraudulent Transactions')
plt.savefig('/home/ubuntu/target_distribution.png')

# Visualization: Correlation Matrix (subset of features)
plt.figure(figsize=(12, 10))
sns.heatmap(df.iloc[:, :15].corr(), annot=False, cmap='coolwarm')
plt.title('Correlation Matrix (First 15 Features)')
plt.savefig('/home/ubuntu/correlation_matrix.png')

# 3. Data Preprocessing
print("\n--- Step 2: Data Preprocessing ---")
X = df.drop(['Class'], axis=1)
y = df['Class']

# Data Splitting
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")

# Data Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Data scaling completed.")

# 4. Model Implementation and Evaluation
results = []

def evaluate_model(model, name, X_t, y_t, X_v, y_v):
    print(f"\nTraining {name}...")
    model.fit(X_t, y_t)
    y_pred = model.predict(X_v)

    acc = accuracy_score(y_v, y_pred)
    prec = precision_score(y_v, y_pred, zero_division=0)
    rec = recall_score(y_v, y_pred, zero_division=0)
    f1 = f1_score(y_v, y_pred, zero_division=0)
    cm = confusion_matrix(y_v, y_pred)

    print(f"{name} Evaluation:")
    print(classification_report(y_v, y_pred, zero_division=0))

    # Save Confusion Matrix Plot
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix: {name}')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.savefig(f'/home/ubuntu/cm_{name.lower().replace(" ", "_")}.png')

    return {
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1
    }

# Step 3: Decision Tree
dt_results = evaluate_model(DecisionTreeClassifier(random_state=42), "Decision Tree", X_train_scaled, y_train, X_test_scaled, y_test)
results.append(dt_results)

# Step 4: Random Forest
rf_results = evaluate_model(RandomForestClassifier(n_estimators=50, random_state=42), "Random Forest", X_train_scaled, y_train, X_test_scaled, y_test)
results.append(rf_results)

# Step 5: AdaBoost
ada_results = evaluate_model(AdaBoostClassifier(n_estimators=50, random_state=42), "AdaBoost", X_train_scaled, y_train, X_test_scaled, y_test)
results.append(ada_results)

# Step 6: Stacking
base_models = [
    ('dt', DecisionTreeClassifier(max_depth=5, random_state=42)),
    ('lr', LogisticRegression(solver='liblinear', random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=5))
]
stacking_model = StackingClassifier(estimators=base_models, final_estimator=LogisticRegression(solver='liblinear', random_state=42))
stack_results = evaluate_model(stacking_model, "Stacking Classifier", X_train_scaled, y_train, X_test_scaled, y_test)
results.append(stack_results)

# Step 8: Comparison Table
comparison_df = pd.DataFrame(results)
print("\n--- Model Comparison ---")
print(comparison_df)
comparison_df.to_csv('/home/ubuntu/model_comparison.csv', index=False)